# 개인 구매이력–후보상품 적합 M2 체크포인트 진단

기존 Dunnhumby seed 42 M1·M2 체크포인트를 다시 학습하지 않고 분석합니다.

1. M1 / ID-only / ID+N / ID+V / full 성과
2. ID·N·V 블록의 실제 후보점수 표준편차와 평균 절댓값
3. 저·중·고CLV별 정답상품의 1–10 / 11–20 / 21–50 / 50위 밖 이동
4. M2가 Top-10으로 새로 올린 상품과 밀어낸 상품의 구매고객 수·가격 특성

이 노트북은 설명용 사후 진단이며 checkpoint 선택·모형 학습·하이퍼파라미터 조정을 하지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '2b9a4ec2880e2ccf968ea11871518fe65165850d'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA

import subprocess
actual_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
print('진단 코드 버전:', actual_sha)


In [ ]:
import json
from lightgcn_clv_history_item_fit_diagnostic import (
    configure_history_item_fit_diagnostic,
    preflight_summary,
    run_history_item_fit_diagnostic,
)

cfg = configure_history_item_fit_diagnostic(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_nv_personal_history_candidate_fit_historical_screen_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
    eval_batch_size=32,
    top_product_examples=20,
)
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))


In [ ]:
report = run_history_item_fit_diagnostic(cfg)


In [ ]:
from IPython.display import display

print('1) M1 및 ID/N/V 블록별 성과')
display(report['view_metrics'])
print('2) 블록별 실제 점수 영향력')
display(report['score_strength'])
print('3) CLV 구간별 정답상품 순위 이동')
display(report['rank_transition'])
print('4) 추천 역할별 상품 인기도·가격 특성')
display(report['item_role_summary'])
print('5) 새로 올라오거나 밀려난 실제 상품 예시')
display(report['product_examples'])
print('저장 파일:', json.dumps(report['paths'], ensure_ascii=False, indent=2))
